# 第3章 编码注意力机制
## 3.4 实现带可训练权重的自注意力机制
### 3.4.2 实现一个简化的自注意力Python类
**代码清单3-1 一个简化的自注意力类**

SelfAttention_v1:

初始化：使用nn.Parameter和torch.rand定义query/key/value等三个权重矩阵。

前向传播：1.计算queries/keys/values三个矩阵的值；2.计算注意力分数；3.计算注意力权重；4.计算上下文向量矩阵；

In [2]:
import torch
import torch.nn as nn

In [1]:
import torch
import torch.nn as nn

# 输入文本示例
inputs = torch.tensor(
    [[0.43, 0.15, 0.89],
     [0.55, 0.87, 0.66],
     [0.57, 0.85, 0.64],
     [0.22, 0.58, 0.33],
     [0.77, 0.25, 0.10],
     [0.05, 0.80, 0.55]]
)

# 通过复制输入文本示例来模拟批量输入
batch_inputs = inputs.repeat(2, 1, 1)
print(f"batch_inputs:{batch_inputs}")
print(f"batch_inputs.shape:{batch_inputs.shape}")

batch = torch.stack((inputs, inputs), dim=0)

batch_inputs:tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])
batch_inputs.shape:torch.Size([2, 6, 3])


In [ ]:
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        queries = x @ self.W_query
        keys = x @ self.W_key
        values = x @ self.W_value

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec

In [ ]:
if __name__ == "__main__":
    torch.manual_seed(123)
    d_in = inputs.shape[-1]
    d_out = 2
    sa1 = SelfAttention_v1(d_in, d_out)
    print(sa1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


**代码清单3-2 使用PyTorch线性层的自注意力类**

SelfAttention_v2:

初始化：使用nn.Linear定义三个权重矩阵

前向传播：同上

In [ ]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out)
        self.W_key = nn.Linear(d_in, d_out)
        self.W_value = nn.Linear(d_in, d_out)
    
    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)
        
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec

In [ ]:
if __name__ == "__main__":
    torch.manual_seed(123)
    d_in = inputs.shape[-1]
    d_out = 2
    sa2 = SelfAttention_v2(d_in, d_out)
    print(sa2(inputs))

tensor([[0.1059, 0.9296],
        [0.1144, 0.9353],
        [0.1143, 0.9353],
        [0.1181, 0.9369],
        [0.1138, 0.9343],
        [0.1188, 0.9375]], grad_fn=<MmBackward0>)


## 3.5 利用因果注意力隐藏未来词汇
因果注意力（也称为**掩码注意力**）是一种特殊的自注意力形式。它限制模型在处理任何给定词元时，只能基于序列中的**先前和当前输入**来计算注意力分数，而标准的自注意力机制可以**一次性访问整个输入序列**。

### 3.5.1 因果注意力的掩码实现 

In [ ]:
if __name__ == "__main__":
    # 复用3.4.2节中的SelfAttention_v2对象的查询权重矩阵和键权重矩阵
    torch.manual_seed(123)
    d_in = inputs.shape[-1]
    d_out = 2
    sa2 = SelfAttention_v2(d_in, d_out)

    queries = sa2.W_query(inputs)
    keys = sa2.W_key(inputs)
    values = sa2.W_value(inputs)

    attn_scores = queries @ keys.T
    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
    print(f"attn_weights:{attn_weights}")

    # 法一：均值归一化
    # 使用PyTorch的tril函数来创建一个对角线以上元素为0的掩码
    context_length = attn_scores.shape[0]   # 注意力分数（输入向量inputs）的第一个维度
    mask_simple = torch.tril(torch.ones(context_length, context_length))
    print(f"mask_simple:{mask_simple}")

    # 将掩码矩阵和注意力权重矩阵相乘，使对角线上方的值变为0
    masked_simple = attn_weights * mask_simple
    print(f"masked_simple:{masked_simple}")

    # 重新归一化注意力权重，使每一行的总和再次为1，通过将每行中的每个元素除以每行中的和来实现
    row_sums = masked_simple.sum(dim=-1,keepdim=True)
    masked_simple_norm = masked_simple / row_sums
    print(f"masked_simple_norm:{masked_simple_norm}")

    # 使用归一化后的掩码矩阵计算上下文向量矩阵
    context_vecs_simple = masked_simple_norm @ values
    print(f"context_vecs_simple:{context_vecs_simple}")

    # 法二：softmax归一化
    # 通过创建一个对角线以上是1的掩码，并将这些1替换为负无穷大（-inf）值，来实现这种更高效的掩码“方法”
    mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
    masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
    print(f"masked:{masked}")

    # 对这些掩码结果应用softmax函数，得到归一化的注意力权重
    attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
    print(f"attn_weights:{attn_weights}")

    # 计算上下文向量矩阵
    context_vecs = attn_weights @ values
    print(f"context_vecs:{context_vecs}")


### 3.5.2 利用dropout掩码额外的注意力权重

dropout技术通过在**训练过程中**随机忽略一些隐藏层单元来有效地“丢弃”它们，这种方法有助于**减少模型对特定隐藏层单元的依赖**，从而避免过拟合。需要强调的是，dropout仅在训练期间使用，训练结束后会被取消。

在Transformer架构中，一些包括GPT在内的模型通常会在两个特定时间点使用注意力机制中的dropout：**一是计算注意力权重之后，二是将这些权重应用于值向量之后**。其中在**计算注意力权重之后**应用dropout掩码，是实践中更常见的做法。

In [ ]:
# 创建一个dropout层
torch.manual_seed(123)
dropout = torch.nn.Dropout(p=0.5)
example = torch.ones(6, 6)
print(f"dropout_example:{dropout(example)}")

In [ ]:
# 对注意力权重矩阵进行dropout操作
torch.manual_seed(123)
print(f"dropout_attn_weights:{dropout(attn_weights)}")

### 3.5.3 实现一个简化的因果注意力类

**任务**：把因果注意力和dropout修改应用到SelfAttention Python类中；首先确保代码可以处理包含多个样本的批次，以便CausalAttention类能够支持数据加载器产生的批量输出。

In [ ]:
# 为了简单起见，可以通过复制输入文本示例来模拟批量输入
batch = torch.stack((inputs, inputs), dim=0)
print(f"batch:{batch}")
print(f"batch.shape:{batch.shape}")
# 生成一个三维向量，其中包含两个输入文本，每个文本有6个词元，每个词元是一个三维的嵌入向量

In [ ]:
class SelfAttention_v3(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout=0.5, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        b, num_tokens, d_in = x.shape
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_scores_dropout = self.dropout(attn_scores)
        # attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = torch.softmax(attn_scores_dropout / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec

In [ ]:
if __name__ == "__main__":
    torch.manual_seed(123)
    d_in = batch.shape[-1]
    d_out = 2
    context_length = batch.shape[1]
    sa3 = SelfAttention_v3(d_in, d_out,context_length, dropout=0.5)
    context_vecs = sa3(batch)
    print(f"context_vecs:{context_vecs}")
    print(f"context_vecs.shape:{context_vecs.shape}")

In [7]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout=0.5, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # self.mask = torch.triu(torch.ones(num_tokens, num_tokens), diagonal=1)
        attn_scores = queries @ keys.transpose(1,2)
        attn_scores.masked_fill_(self.mask.bool(), -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vecs = attn_weights @ values
        return context_vecs

In [ ]:
if __name__ == "__main__":
    torch.manual_seed(123)
    batch = torch.stack((inputs, inputs), dim=0)
    d_in = batch.shape[-1]
    d_out = 2
    context_length = batch.shape[1]
    causal_attn = CausalAttention(d_in, d_out, context_length, dropout=0.5,)
    context_vecs = causal_attn(batch)
    print(f"context_vecs:{context_vecs}")
    print(f"context_vecs.shape:{context_vecs.shape}")

## 3.6 将单头注意力扩展到多头注意力
本节将把先前实现的因果注意力类扩展到多个头上，这也被称为**多头注意力**

“多头”这一术语指的是将注意力机制分成多个“头”，每个“头”独立工作。在这种情况下，单个因果注意力模块可以被看作单头注意力，因为它只有一组注意力权重按顺序处理输入。

首先将直观地通过堆叠多个CausalAttention模块来构建多头注意力模块。然后，将用一种更复杂但计算上更高效的方式来实现这个多头注意力模块。

### 3.6.1 叠加多个单头注意力层
在实际操作中，实现多头注意力需要构建多个自注意力机制的实例，每个实例都有其独立的权重，然后将这些输出进行合成。虽然这种方法的计算量可能会非常大，但它对诸如基于transformer的大语言模型之类的模型的复杂模式识别是非常重要的。

多头注意力的主要思想是**多次（并行）运行注意力机制**，每次使用学到的不同的线性投影——这些投影是通过将输入数据（比如注意力机制中的查询向量、键向量和值向量）乘以权重矩阵得到的。

在代码中，可以通过实现一个简单的MultiHeadAttentionWrapper类来达到这一目标，MultiHeadAttentionWrapper类堆叠了多个之前实现的CausalAttention模块实例。

In [8]:
# 一个实现多头注意力的封装类
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList([CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) for _ in range(num_heads)])

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [ ]:
if __name__ == "__main__":
    torch.manual_seed(123)
    context_length = batch.shape[1]
    d_in, d_out = 3, 2 
    mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)
    context_vecs = mha(batch)
    print(f"context_vecs:{context_vecs}")
    print(f"context_vecs.shape:{context_vecs.shape}")

In [3]:
# nn.ModuleList实例
import torch
import torch.nn as nn

class MyModule(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linears = nn.ModuleList([nn.Linear(10,10) for i in range(10)])
        
    def forward(self, x):
        for i, l in enumerate(self.linears):
            x = self.linears[i // 2](x) + l(x)
        return x

if __name__ == "__main__":
    mymodule = MyModule()
    inputs = torch.linspace(1,10,10)
    inputs_rep = inputs.repeat(10,1)
    print(mymodule(inputs_rep))

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        queries = queries.transpose(1 ,2)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1,2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)
        return context_vec

In [12]:
if __name__ == "__main__":
    torch.manual_seed(123)
    batch_size, context_length, d_in = batch.shape
    d_out = 2
    mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
    context_vecs = mha(batch)
    print(f"context_vecs:{context_vecs}")
    print(f"context_vecs.shape:{context_vecs.shape}")


In [4]:
mah = MultiHeadAttention(3,2,6,0.0,2)
print(mah)